[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tasmia008/Complete_Guidelines_LLM_FineTuning/blob/main/01_foundations.ipynb)

# Foundations — tokenizers, templates, loss, optimizer, hyperparams, eval, precision, data

> **Fine-tuning series — 01 of 26.** A standalone notebook in a set; see `00_README.md` for the full index and order. This notebook is self-contained: run **Setup**, then the rest.


## Setup (run first)

Self-contained: imports, model id, device flags, and a tiny inline dataset so this notebook runs on its own.

In [ ]:
# Environment check — record these versions with any results you report.
import importlib
for lib in ["torch", "transformers", "peft", "trl", "datasets",
            "bitsandbytes", "accelerate", "unsloth", "adapters"]:
    try:
        m = importlib.import_module(lib)
        print(f"{lib:14s} {getattr(m, '__version__', '?')}")
    except Exception as e:
        print(f"{lib:14s} not installed ({type(e).__name__})")

In [ ]:
# !pip install -U transformers peft trl bitsandbytes datasets accelerate
# !pip install unsloth   # only for the Unsloth section (CUDA only)
import gc, torch
from datasets import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"     # small + fast; swap for any causal LM
device = ("cuda" if torch.cuda.is_available()
          else "mps" if torch.backends.mps.is_available() else "cpu")
BF16_OK = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
FP16_OK = torch.cuda.is_available() and not BF16_OK   # fp16 needs a GPU
print("device:", device, "| bf16:", BF16_OK)

def cleanup():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

In [ ]:
# (a) Instruction data -> for SFT / LoRA / QLoRA / Unsloth / instruction / prompt tuning
instructions = [
    {"instruction": "Give the capital of France.", "output": "Paris."},
    {"instruction": "What is 7 times 8?", "output": "56."},
    {"instruction": "Translate 'good morning' to Spanish.", "output": "Buenos días."},
    {"instruction": "Name the largest planet.", "output": "Jupiter."},
    {"instruction": "Define photosynthesis in one line.",
     "output": "Plants converting sunlight, water, and CO2 into energy and oxygen."},
] * 8   # repeat just to have enough steps in the demo

# (b) Preference data -> for DPO (prompt + chosen/rejected, no reward model)
preferences = [
    {"prompt": "Explain gravity to a child.",
     "chosen": "Gravity is the pull that brings things down to the ground.",
     "rejected": "Gravity is a tensor field obeying the Einstein equations."},
    {"prompt": "Recommend a book.",
     "chosen": "Try 'The Hobbit' — a fun, easy adventure.",
     "rejected": "Read whatever, I don't care."},
] * 16

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def to_chat_text(ex):
    msgs = [{"role": "user", "content": ex["instruction"]},
            {"role": "assistant", "content": ex["output"]}]
    return {"text": tokenizer.apply_chat_template(msgs, tokenize=False)}

sft_ds = Dataset.from_list([to_chat_text(e) for e in instructions])
pref_ds = Dataset.from_list(preferences)
print(sft_ds[0]["text"][:160])

## Foundations — what every method assumes

*New to fine-tuning? Read this first; skip if these are second nature.* Every section
below leans on the ideas here. The demos use the `tokenizer`/`MODEL_ID` and the inline
datasets from **Setup**.

### 1. Tokenizers

A model never sees text — it sees integer **token IDs**. The tokenizer splits text into
subword **tokens** and maps each to an ID in a fixed **vocabulary**, plus special tokens
(beginning/end-of-sequence, padding). Training and generation both operate on these IDs;
`decode` maps them back to text.

In [ ]:
text = "Fine-tuning adapts a model to a task."
enc = tokenizer(text)
print("IDs    :", enc["input_ids"])
print("tokens :", tokenizer.convert_ids_to_tokens(enc["input_ids"]))
print("decoded:", tokenizer.decode(enc["input_ids"]))
print("vocab size:", tokenizer.vocab_size, "| eos:", repr(tokenizer.eos_token), tokenizer.eos_token_id)

### 2. Chat templates

Instruct models were trained on conversations wrapped in a **specific format** with role
markers (Qwen uses `<|im_start|>user … <|im_end|>`). Feed raw text and the model
underperforms — you must match that format. `apply_chat_template` builds it from a list of
`{role, content}` messages. `add_generation_prompt=True` leaves the assistant turn *open*
so the model continues it (used at inference); for training you include the assistant's
answer instead.

In [ ]:
messages = [{"role": "user", "content": "What is LoRA?"}]
print("--- inference (assistant turn left open) ---")
print(tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True))

train_msgs = messages + [{"role": "assistant", "content": "A low-rank adapter."}]
print("--- training (assistant answer included) ---")
print(tokenizer.apply_chat_template(train_msgs, tokenize=False))

### 3. Optimizer state — why full fine-tuning is memory-heavy

Training updates weights with an optimizer. **AdamW** keeps, *per parameter*, two running
averages: the first moment `m` (momentum) and second moment `v` (variance). So beyond the
weight you also store a gradient, an fp32 master copy, `m`, and `v` — roughly **16
bytes/param** in mixed precision. That overhead, not the weights, is what blows up memory.

This is exactly why LoRA/QLoRA save: the base is **frozen**, so gradients, `m`, and `v`
exist only for the `<1%` of parameters in the adapters.

In [ ]:
import torch
lin = torch.nn.Linear(1000, 1000)
opt = torch.optim.AdamW(lin.parameters(), lr=1e-3)
lin(torch.randn(8, 1000)).sum().backward()
opt.step()

state = opt.state[lin.weight]
print("AdamW stores per weight:", [k for k in state if torch.is_tensor(state[k])])  # exp_avg=m, exp_avg_sq=v

P = 7e9                                   # 7B params, ~16 bytes/param overhead
print(f"\nFull FT overhead : {P*16/1e9:.0f} GB  (grad + fp32 master + m + v, all params)")
trainable = 0.003 * P                     # ~0.3% with LoRA
print(f"LoRA overhead    : {trainable*16/1e9:.2f} GB  (on ~{trainable/1e6:.0f}M adapter params)")
print(f"  + frozen base in bf16: {P*2/1e9:.0f} GB  ->  LoRA total ~14.3 GB vs full ~112 GB")
del lin, opt

### 4. q / v projections — where LoRA attaches

Inside each transformer layer, **self-attention** has four linear maps: `q_proj`, `k_proj`,
`v_proj` build the queries/keys/values from the input, and `o_proj` combines the result.
The MLP block adds `gate_proj`/`up_proj`/`down_proj`. LoRA usually targets `q_proj` and
`v_proj` (often all of them) — that's where adaptation has the most leverage.

Two gotchas you'll meet later: under **grouped-query attention** `k_proj`/`v_proj` are
*smaller* than `q_proj` (fewer key/value heads), and some models **fuse** q,k,v into one
matrix (`query_key_value`, `Wqkv`) — which is why §8's auto-discovery matters: target the
wrong name and the adapter attaches to nothing.

In [ ]:
from transformers import AutoModelForCausalLM
m = AutoModelForCausalLM.from_pretrained(MODEL_ID)
attn = m.model.layers[0].self_attn          # first layer's attention block
for name in ["q_proj", "k_proj", "v_proj", "o_proj"]:
    w = getattr(attn, name).weight
    print(f"{name:8s} weight {tuple(w.shape)}  (out, in)")
print("note: k_proj/v_proj smaller than q_proj => grouped-query attention")
del m

### 5. The loss function — what training actually minimizes

A causal LM predicts the **next token**. At each position it outputs a vector of **logits**
over the vocabulary; `softmax` turns those into probabilities; the loss is the **negative
log-likelihood** of the token that actually came next — i.e. **cross-entropy**:

$$L = -\frac{1}{N}\sum_{t} \log p_\theta\!\left(x_{t+1}\mid x_{\le t}\right).$$

Labels are just the inputs **shifted by one** (position $t$ predicts token $t{+}1$); HF does
this shift internally when you pass `labels`. Positions with label **`-100`** are *ignored* —
that's the mechanism behind completion-only / instruction masking. (Preference methods like
DPO and reward modeling replace this next-token loss with a *preference* loss instead.)

In [ ]:
import numpy as np
def softmax(z): e = np.exp(z - z.max()); return e / e.sum()

logits = np.array([2.0, 0.5, 1.0, -1.0, 0.2])   # model's scores over a 5-token vocab
target = 0                                       # the token that actually came next
p = softmax(logits)
print("probs       :", np.round(p, 3))
print("p(target)   :", round(p[target], 3), "-> cross-entropy = -log p =", round(-np.log(p[target]), 3))

# averaged over a sequence, skipping masked (-100) positions
labels = np.array([0, 2, -100, 4])               # -100 = prompt tokens we don't train on
seq = np.array([[2,.5,1,-1,.2], [.1,.2,3,0,1], [1,1,1,1,1], [0,0,0,0,5]], float)
losses = [-np.log(softmax(lg)[y]) for lg, y in zip(seq, labels) if y != -100]
print("mean loss over non-masked tokens:", round(float(np.mean(losses)), 3))

### 6. Hyperparameters — the knobs in every training cell

The numbers you see repeated in `SFTConfig`/`TrainingArguments`:

- **learning rate** — step size. Rough defaults: full FT `~2e-5`, LoRA/QLoRA `~2e-4`, DPO `~5e-6`.
- **batch size** (`per_device_train_batch_size`) — examples per step, per GPU.
- **gradient accumulation** — accumulate grads over several micro-batches before stepping, to
  *simulate* a big batch under tight memory: `effective batch = per_device_bs × grad_accum × #GPUs`.
- **epochs vs steps** — an *epoch* is one full pass over the data; `max_steps` caps the number
  of weight updates regardless of epochs (handy for demos).
- **warmup** — ramp the LR up from 0 over the first few steps so early updates don't destabilize.
- **schedule** (`lr_scheduler_type`) — how the LR decays after warmup (cosine or linear are usual).

In [ ]:
import numpy as np, matplotlib.pyplot as plt

total, warmup, peak = 200, 20, 2e-4
def lr_at(step):
    if step < warmup:
        return peak * step / warmup                    # linear warmup
    prog = (step - warmup) / (total - warmup)
    return 0.5 * peak * (1 + np.cos(np.pi * prog))      # cosine decay

plt.figure(figsize=(6, 3))
plt.plot(range(total), [lr_at(s) for s in range(total)])
plt.xlabel("step"); plt.ylabel("learning rate"); plt.title("warmup + cosine decay")
plt.tight_layout(); plt.show()

print("effective batch = per_device_bs x grad_accum x #GPUs  =  2 x 8 x 1 =", 2*8*1)

### 7. Evaluation & overfitting — how you know it worked

Split your data three ways: **train** (fit), **validation** (monitor + tune
hyperparameters), **test** (one final, unbiased number). Watching both losses tells you
what's happening:

- **Underfitting** — train *and* val loss both high: model/data/LR too weak; train more.
- **Overfitting** — train loss keeps falling but **val loss turns back up**: the model is
  memorizing. Fix with **early stopping** (stop at the val-loss minimum), more data, dropout,
  weight decay, or a smaller LoRA rank.

Metrics depend on the task: **perplexity** $=e^{\text{loss}}$ for language modeling (lower is
better), **accuracy/F1** for classification, task-specific scores (BLEU/ROUGE) for generation.

In [ ]:
import numpy as np, matplotlib.pyplot as plt

print("cross-entropy loss 1.6  ->  perplexity = exp(1.6) =", round(float(np.exp(1.6)), 2))

ep = np.arange(1, 21)
train = 2.5 * np.exp(-0.18 * ep) + 0.2
val   = train + 0.02 * np.maximum(ep - 6, 0) ** 2          # val turns up = overfitting
best  = int(ep[np.argmin(val)])

plt.figure(figsize=(6, 3))
plt.plot(ep, train, label="train loss"); plt.plot(ep, val, label="val loss")
plt.axvline(best, ls="--", c="gray"); plt.text(best + 0.2, val.max() * 0.7, f"early stop @ {best}")
plt.xlabel("epoch"); plt.ylabel("loss"); plt.legend(); plt.title("overfitting: val loss turns up")
plt.tight_layout(); plt.show()

### 8. Precision — fp32 / fp16 / bf16 / 4-bit

How many bits each weight uses trades memory/speed against numerical stability:

- **fp32** (4 B) — full precision, most stable, most memory. The safe fallback.
- **fp16** (2 B) — half precision; fast and light but a *narrow range* → can overflow/underflow,
  so it needs loss-scaling. Use on older GPUs (T4/V100).
- **bf16** (2 B) — same exponent **range** as fp32 (so stable) with less mantissa precision;
  the default on Ampere+ (A100/RTX 30xx+). This is what the `BF16_OK` flag gates.
- **mixed precision** — keep an fp32 master copy, compute in bf16/fp16: speed *and* stability.
- **4-bit (NF4)** — *storage only*, the QLoRA trick (see the math deep-dive).

In [ ]:
import numpy as np
for dt in (np.float32, np.float16):
    info = np.finfo(dt)
    print(f"{dt.__name__:9s} {info.bits:2d} bits | max {info.max:.2e} | ~{abs(int(np.log10(info.eps)))} decimal digits")
print("bfloat16   16 bits | max ~3.4e38 (fp32 range!) | ~3 decimal digits  -> stable + light")
print("\nrule of thumb: bf16 on A100/RTX30xx+, fp16 on T4/V100, fp32 if training is unstable")

### 9. Data formats by objective

Each axis wants a different data shape — get this right and most trainers "just work":

| Objective | Trainer | Each row looks like |
|---|---|---|
| Instruction / SFT | SFTTrainer | `{instruction, output}` or chat `messages` → one `text` string |
| Preference | DPO / ORPO | `{prompt, chosen, rejected}` |
| Unpaired feedback | KTO | `{prompt, completion, label: bool}` |
| Reward model | RewardTrainer | `{chosen, rejected}` (full texts) |
| Reward-function RL | GRPO | just `{prompt}` + a `reward_funcs` (no target label) |

In [ ]:
# Same rows, reshaped for each objective (uses Setup's inline data).
print("SFT (instruction):", instructions[0])
print("Preference (DPO)  :", preferences[0])
print("Unpaired (KTO)    :", {"prompt": preferences[0]["prompt"],
                              "completion": preferences[0]["chosen"], "label": True})
print("Reward pair       :", {"chosen": "<prompt> " + preferences[0]["chosen"],
                              "rejected": "<prompt> " + preferences[0]["rejected"]})
print("GRPO (prompt only):", {"prompt": instructions[0]["instruction"]}, "+ a reward function")